In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

In [ ]:
import sys
from pathlib import Path

SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
sys.path.insert(0, SRC)

import torch
from layer import L_DG, L_CA3

torch.manual_seed(42)
dg = L_DG(n_input=15, n_DG=100, k_frac=0.01, ecin_frac=0.25, use_euler=True)
dg.W.data = torch.randn(15, 100) * 0.1

a_A = torch.zeros(15); a_A[0] = 1.0; a_A[1] = 0.9
a_B = torch.zeros(15); a_B[1] = 1.0; a_B[2] = 0.9

dg.reset(); act_A = dg(a_A).clone()
dg.reset(); act_B = dg(a_B).clone()

## Step 4: L_CA3 — CA3 Field

**Role**: Pattern completion via recurrent attractor dynamics.
A partial cue from DG activates a partial CA3 pattern; recurrent connections reinstate the full stored pattern.

**Key parameters (Schapiro 2017 §2.a.iii)**:
- `dg_frac = 0.05`: 5% sparse mossy fibre from DG (much sparser than ECin→DG)
- `k_frac = 0.10`: ~10% active — less sparse than DG to allow overlap for completion
- `W_rec`: fully connected CA3→CA3 (Hopfield attractor)

**Understanding check**: What does W_rec do on the first trial of a new item?  
→ W_rec starts at zero → no recurrent input → CA3 settles based on DG input alone (feedforward only).
After the first CHL update W_rec begins to store the pattern. After several trials the attractor is stable enough that a partial DG cue reinstates the full CA3 pattern.

In [ ]:
# Instantiate and inspect structure
ca3 = L_CA3(n_DG=100, n_CA3=50, k_frac=0.10, dg_frac=0.05, use_euler=True)

print(f"W_ff shape     : {ca3.W_ff.shape}")          # (100, 50)
print(f"mask_ff shape  : {ca3.mask_ff.shape}")        # (100, 50)
print(f"mask_ff density: {ca3.mask_ff.mean():.3f}")   # ~0.05
print(f"W_rec shape    : {ca3.W_rec.shape}")           # (50, 50) — fully connected
print(f"n_active target: {max(1, int(ca3.k_frac * ca3.n_CA3))}")  # 5 units

In [ ]:
# Forward pass — sparsity check
torch.manual_seed(0)
ca3.W_ff.data  = torch.randn(100, 50) * 0.1
ca3.W_rec.data = torch.randn(50, 50)  * 0.05

ca3.reset()
act_ca3 = ca3(act_A)

n_active = (act_ca3 > 0).sum().item()
print(f"Active CA3 units: {n_active} / {ca3.n_CA3} = {n_active / ca3.n_CA3:.3f}")
print(f"Expected ~10% = {int(ca3.k_frac * ca3.n_CA3)} units")

In [ ]:
# Recurrent dynamics: pattern stored then retrieved from zero DG input

# Step 1: settle for 25 cycles to get stable CA3 pattern
ca3.reset()
for _ in range(25):
    stored = ca3(act_A)
stored = stored.clone()
print(f"Stored pattern active units: {(stored > 0).sum().item()}")

# Step 2: one CHL update to write pattern into W_rec
ca3.update_weights(
    a_DG_minus=torch.zeros(100), a_DG_plus=act_A,
    a_CA3_minus=torch.zeros(50), a_CA3_plus=stored,
    lr=0.4,
)

# Step 3: zero DG input — W_rec alone should partially reinstate
ca3.reset()
act_from_rec = ca3(torch.zeros(100))
overlap = ((stored > 0) & (act_from_rec > 0)).sum().item()
print(f"Pattern completion from zero DG input:")
print(f"  active in stored  : {(stored > 0).sum().item()}")
print(f"  active in recalled: {(act_from_rec > 0).sum().item()}")
print(f"  overlap           : {overlap}")
print("(overlap increases with more CHL updates)")

In [ ]:
# CHL weight update — mask on W_ff, no mask on W_rec
ca3_test = L_CA3(n_DG=100, n_CA3=50, k_frac=0.10, dg_frac=0.05)
torch.manual_seed(1)
ca3_test.W_ff.data  = torch.randn(100, 50) * 0.1
ca3_test.W_rec.data = torch.randn(50, 50)  * 0.05

ca3_test.reset(); a_ca3_m = ca3_test(act_A).clone()
ca3_test.reset(); a_ca3_p = ca3_test(act_B).clone()

Wff_before  = ca3_test.W_ff.data.clone()
Wrec_before = ca3_test.W_rec.data.clone()

ca3_test.update_weights(act_A, act_B, a_ca3_m, a_ca3_p, lr=0.4)

dWff  = ca3_test.W_ff.data  - Wff_before
dWrec = ca3_test.W_rec.data - Wrec_before

print(f"W_ff  changed at masked   positions: {(dWff[ca3_test.mask_ff.bool()] != 0).sum().item()}")
print(f"W_ff  changed at unmasked positions: {(dWff[~ca3_test.mask_ff.bool()] != 0).sum().item()}")
print(f"W_rec changed ({ca3_test.n_CA3}×{ca3_test.n_CA3} = {ca3_test.n_CA3**2} entries): {(dWrec != 0).sum().item()}")
print("W_ff unmasked must stay zero. W_rec fully updated (no mask).")